# Customer Churn Analysis

## PDSML Day 2 - Importing Libraries and Dataset

### What is a Python Library

A Python library is a collection of ready-made code (functions, classes, and modules) that helps you do specific tasks easily — like math, data analysis, or web work — without writing everything from scratch.

### Importing essential library

In [ ]:
#Exploratory Descriptive Analysis(EDA), Dataset import
import pandas as pd
#Mathametical Operation
import numpy as np
#To devide the training set and test set
from sklearn.model_selection import train_test_split #scikit-learn -> machine learning package/library ... model_selection is the submodule. and in this model selection there is an funcion called train_test_split.. by this we just import train_test_split
#imbalance learning -> The result are should be more or less 50%. this will do the synthetic oversampling. minority group create more. and it is and synthetic process.
from imblearn.over_sampling import SMOTE #SMOTE -> Synthetic Minority Oversampling Technique
#if someone churn, why he did this, which indepancnce factor is the most important. it determine to identifying the indepandance factor
import shap

## PDSML Day 3 - Data Preprocessing (Part-1)

### import dataset.


First upload the dataset in the colab and then copy the name or path.

In [ ]:
dataset = pd.read_csv("Customer-Churn.csv")

### Checking the head of the dataset

In [ ]:
#To show the first 5 row of the dataset
dataset.head()
#If you want to just see first n dataset
#dataset.head(n)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Checking the Shape  of the Dataset

In [ ]:
dataset.shape
#Give --> (Rows, columns)

(7043, 21)

In [ ]:
dataset.head(), dataset.shape

(   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
 0  7590-VHVEG  Female              0     Yes         No       1           No   
 1  5575-GNVDE    Male              0      No         No      34          Yes   
 2  3668-QPYBK    Male              0      No         No       2          Yes   
 3  7795-CFOCW    Male              0      No         No      45           No   
 4  9237-HQITU  Female              0      No         No       2          Yes   
 
       MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
 0  No phone service             DSL             No  ...               No   
 1                No             DSL            Yes  ...              Yes   
 2                No             DSL            Yes  ...               No   
 3  No phone service             DSL            Yes  ...              Yes   
 4                No     Fiber optic             No  ...               No   
 
   TechSupport StreamingTV StreamingMovies      

### Checking Imbalanced Dataset

In [ ]:
#just want to watch the one column
dataset["Churn"]
#count the value of the specific column
dataset["Churn"].value_counts()
#See in proportion in percentage.. Num/Total
dataset["Churn"].value_counts(normalize= True)

,proportion
Churn,
No,0.73463
Yes,0.26537


### Data Cleaning

#### Detecting the erroneous numeric columns by calculating the mean

**Task:-**

    You need to check a independent column in the dataset for errors.

**What to check:-**

**1. Missing values**

    Find if the column has any NA or empty values.

**2. Spacing mistakes**

    Check if the data has extra spaces at the start or end.

**3. Mean calculation**

    Try to calculate the mean of the column.

    If the mean is not shown, that means the column has an error (maybe text, spaces, or wrong data type).

In [ ]:
print(dataset["tenure"].mean())
print(dataset["MonthlyCharges"].mean())

32.37114865824223
64.76169246059918


In [ ]:
#to check there any missing value in the column.
#print(dataset["TotalCharges"].mean())

In [ ]:
#as the Regular expression only work on the string you first need to convert the number into string.
import re

for i in range(len(dataset)):
  value = str(dataset["TotalCharges"].iloc[i])
  if not re.match(r'^\d+(\.\d+)?$', value):
    print(f"The error cell is: {i}")


The error cell is: 488
The error cell is: 753
The error cell is: 936
The error cell is: 1082
The error cell is: 1340
The error cell is: 3331
The error cell is: 3826
The error cell is: 4380
The error cell is: 5218
The error cell is: 6670
The error cell is: 6754


In [ ]:
# To see a specific value
dataset["TotalCharges"].iloc[3331]

' '

#### Converting the string entries to numeric and NaN

In [ ]:
# Not a Number --> NaN
# Make the string to numeric
# if there the data cannot convert into the the numaric then it will convert into NaN
# error = "coerce" -> when find the string make it NaN. if any value can’t be converted (like text or blank), make it NaN instead of giving an error.
dataset["TotalCharges"] = pd.to_numeric(dataset["TotalCharges"], errors = "coerce")

In [ ]:
#checking it after conversion
print(dataset["TotalCharges"].iloc[3331])

nan


In [ ]:
#checking mean of the Total Charges
print(dataset["TotalCharges"].mean())
#as it shows the mean. that's means the column is clean now.

2283.3004408418656


Attribute → stores information → no parentheses

Method / Function → performs an action → needs parentheses()

#### Dropping the nan-containing rows from the dataframe

In [ ]:
dataset.shape

(7043, 21)

In [ ]:
# Now we will delete the data which is missing.
# In machine learning we will not fill the missing value. As the dataset is huge. We will not do data manupulation.
dataset = dataset.dropna(subset = ["TotalCharges"]).copy()

In [ ]:
dataset.shape

(7032, 21)

## PDSML Day 4 - Data preprocessing (Part-2)

### Removing unneccessary features

In [ ]:
dataset.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


When the value is in the ["items1", "items2",....]-->square bracket then it is called list.


In [ ]:
# To see the dataset columns
list(dataset.columns)
# When the value is in the ["items1", "items2",....]-->square bracket then it is called list.

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

**Concept**


    In machine learning, independent variables are usually stored in a variable named X.

    X is created by selecting all the independent (input) features from the dataset.

    X is also called the feature matrix, because it holds all the columns used to predict the target.



In [ ]:
X = dataset.drop(columns=["customerID", "Churn"])

In [ ]:
list(X.columns)

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges']

In [ ]:
X.shape

(7032, 19)

### Dependent variable decleration

In [ ]:
print(dataset.Churn.head())

0     No
1     No
2    Yes
3     No
4    Yes
Name: Churn, dtype: object


In [ ]:
y = dataset["Churn"]

In [ ]:
y

,Churn
0,No
1,No
2,Yes
3,No
4,Yes
...,...
7038,No
7039,No
7040,No
7041,Yes


The machine learning algorithm cannot work on the string file. Now we need to do Numeric encoding.

In [ ]:
#dictorary --> {'key1':value1, 'key1':value1}
y = y.map({'No':0, 'Yes':1})

In [ ]:
y.head()

,Churn
0,0
1,0
2,1
3,0
4,1


### Spliting the dataset into the trainging set and test set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

1. test_size = 0.2

        20% of data goes to test

        80% goes to train

2. random_state = 42

        Fixes the randomness

        You will always get the same split every time you run it

3. stratify = y

        Keeps the same class ratio in both train and test

In [ ]:
print(X_train.head())

      gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
1413    Male              0     Yes        Yes      65          Yes   
7003    Male              0      No         No      26           No   
3355  Female              0     Yes         No      68          Yes   
4494    Male              0      No         No       3          Yes   
3541  Female              0     Yes         No      49           No   

         MultipleLines InternetService OnlineSecurity OnlineBackup  \
1413               Yes     Fiber optic            Yes          Yes   
7003  No phone service             DSL             No           No   
3355               Yes     Fiber optic             No          Yes   
4494                No     Fiber optic             No          Yes   
3541  No phone service             DSL            Yes           No   

     DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
1413              Yes         Yes          No              No        Tw

In [ ]:
y_train.value_counts(normalize= True)

,proportion
Churn,
0,0.734222
1,0.265778


In [ ]:
y_test.value_counts(normalize= True)

,proportion
Churn,
0,0.734186
1,0.265814


## PDSML Day-5 👉 Data Preprocessing (Part-3)

### One-hot encoding

**Vector Encoding & Dummy Variables**

    Machine learning models cannot read text, so we must convert all categorical text into numbers.

    But we cannot convert categories as 1, 2, 3, 4 because it creates a false ranking (model thinks 4 > 1).

    Categorical data has no order, so we avoid direct numbering.

**Independent Variables Have Two Types**

    Categorical → text labels

    Numerical → numbers

    Both types must be checked before training.

**For Categorical Variables**

    Convert them using Dummy Variables (One-Hot Encoding).

    Each category becomes a new column with 0/1 values.

    Example (Color):
    This is an example of One Hot encoder
| Color_Red | Color_Blue | Color_Green | Color_Yellow |
| --------- | ---------- | ----------- | ------------ |
| 1         | 0          | 0           | 0            |
| 0         | 1          | 0           | 0            |
| 0         | 0          | 1           | 0            |
| 0         | 0          | 0           | 1            |
| 1         | 0          | 0           | 0            |
| 0         | 0          | 1           | 0            |


    Even Binary Categories (like 0/1) Should Be Treated as Categorical

    Example: “Partner: 0 = No, 1 = Yes”

    It is still categorical, not numerical.

    So convert it into dummy columns to avoid wrong interpretation.



Nummeric column -> freature endocing -> scale fix.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

#### Selecting the numeric column and catagorical column

In [ ]:
num_cols = X.select_dtypes(include = np.number).columns.tolist()

In [ ]:
num_cols

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

#### Selecting the categorical columns

In [ ]:
cat_cols = X.select_dtypes(exclude = np.number).columns.tolist()

In [ ]:
cat_cols

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

#### Columns Trasformations

**handle_unknown = "ignore" --> If a new category appears, ignore it (no error).**
    
    handle_unknown="ignore" tells OneHotEncoder to skip any category not seen in training.
    Unknown categories in test data get all zeros in the encoded columns.
    This prevents errors without adding new columns

**sparse output**


In [ ]:
pre = ColumnTransformer(
    transformers = [
        ("cata", OneHotEncoder(handle_unknown = "ignore", sparse_output= False), cat_cols),
        ("numb", "passthrough", num_cols)
    ]
)

In [ ]:
print(pre.fit_transform(X))

[[1.0000e+00 0.0000e+00 0.0000e+00 ... 1.0000e+00 2.9850e+01 2.9850e+01]
 [0.0000e+00 1.0000e+00 1.0000e+00 ... 3.4000e+01 5.6950e+01 1.8895e+03]
 [0.0000e+00 1.0000e+00 1.0000e+00 ... 2.0000e+00 5.3850e+01 1.0815e+02]
 ...
 [1.0000e+00 0.0000e+00 0.0000e+00 ... 1.1000e+01 2.9600e+01 3.4645e+02]
 [0.0000e+00 1.0000e+00 0.0000e+00 ... 4.0000e+00 7.4400e+01 3.0660e+02]
 [0.0000e+00 1.0000e+00 1.0000e+00 ... 6.6000e+01 1.0565e+02 6.8445e+03]]


## PDSML Day-6 👉 Data Preprocessing (Part-4)

### Getting names of encoded categorical columns

In [ ]:
cat_iv_names = pre.named_transformers_["cata"].get_feature_names_out(cat_cols)

In [ ]:
print(cat_iv_names)

['gender_Female' 'gender_Male' 'Partner_No' 'Partner_Yes' 'Dependents_No'
 'Dependents_Yes' 'PhoneService_No' 'PhoneService_Yes' 'MultipleLines_No'
 'MultipleLines_No phone service' 'MultipleLines_Yes'
 'InternetService_DSL' 'InternetService_Fiber optic' 'InternetService_No'
 'OnlineSecurity_No' 'OnlineSecurity_No internet service'
 'OnlineSecurity_Yes' 'OnlineBackup_No' 'OnlineBackup_No internet service'
 'OnlineBackup_Yes' 'DeviceProtection_No'
 'DeviceProtection_No internet service' 'DeviceProtection_Yes'
 'TechSupport_No' 'TechSupport_No internet service' 'TechSupport_Yes'
 'StreamingTV_No' 'StreamingTV_No internet service' 'StreamingTV_Yes'
 'StreamingMovies_No' 'StreamingMovies_No internet service'
 'StreamingMovies_Yes' 'Contract_Month-to-month' 'Contract_One year'
 'Contract_Two year' 'PaperlessBilling_No' 'PaperlessBilling_Yes'
 'PaymentMethod_Bank transfer (automatic)'
 'PaymentMethod_Credit card (automatic)' 'PaymentMethod_Electronic check'
 'PaymentMethod_Mailed check']


### Combining the categorical columns with the numeric columns

In [ ]:
print(num_cols)

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [ ]:
all_iv_names = list(cat_iv_names) + num_cols

In [ ]:
print(all_iv_names)

['gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_No', 'PaperlessBilling_Yes', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'Payment

### Converting the transformed output to a DataFrame

In [ ]:
X_trasformed = pre.fit_transform(X)

In [ ]:
print(X_trasformed)

[[1.0000e+00 0.0000e+00 0.0000e+00 ... 1.0000e+00 2.9850e+01 2.9850e+01]
 [0.0000e+00 1.0000e+00 1.0000e+00 ... 3.4000e+01 5.6950e+01 1.8895e+03]
 [0.0000e+00 1.0000e+00 1.0000e+00 ... 2.0000e+00 5.3850e+01 1.0815e+02]
 ...
 [1.0000e+00 0.0000e+00 0.0000e+00 ... 1.1000e+01 2.9600e+01 3.4645e+02]
 [0.0000e+00 1.0000e+00 0.0000e+00 ... 4.0000e+00 7.4400e+01 3.0660e+02]
 [0.0000e+00 1.0000e+00 1.0000e+00 ... 6.6000e+01 1.0565e+02 6.8445e+03]]


pandas convert the array to the dataframe

In [ ]:
tr_df = pd.DataFrame(X_trasformed, columns = all_iv_names)
pd.set_option("display.max_columns", None)
print(tr_df.head())

   gender_Female  gender_Male  Partner_No  Partner_Yes  Dependents_No  \
0            1.0          0.0         0.0          1.0            1.0   
1            0.0          1.0         1.0          0.0            1.0   
2            0.0          1.0         1.0          0.0            1.0   
3            0.0          1.0         1.0          0.0            1.0   
4            1.0          0.0         1.0          0.0            1.0   

   Dependents_Yes  PhoneService_No  PhoneService_Yes  MultipleLines_No  \
0             0.0              1.0               0.0               0.0   
1             0.0              0.0               1.0               1.0   
2             0.0              0.0               1.0               1.0   
3             0.0              1.0               0.0               0.0   
4             0.0              0.0               1.0               1.0   

   MultipleLines_No phone service  MultipleLines_Yes  InternetService_DSL  \
0                             1.0      

## PDSML Day-8 👉 Training the Baseline Model

### Training the baseline model

1. **Supervised Learning**

    We already know:

    *   Input (independent variables)
    *   Output (dependent variable)
    Data is labeled

    We give the model examples, it learns patterns, and predicts output.


    Two types:

          a) Regression
          Output is continuous / numerical

          Examples:
          Predict house price based on size
          Forecast monthly income
          Predict temperature, rainfall amount
          Estimate crop yield

          b) Classification
          Output is categories / classes (not continuous)

          Examples:

          Spam vs Non-spam email
          Yes/No loan approval
          Disease positive or negative
          Classifying types of fruits (apple, orange, banana)

2. **Unsupervised Learning**

        No dependent variable
        Data has no labels
        Model finds patterns on its own


        Most common example: Clustering
        Customer segmentation (grouping buyers by habits)
        Grouping similar images without label
        Market basket analysis (which items are often bought together)

3. **Reinforcement Learning**

        Learning by trial and error
        System gets reward for right actions and punishment/negative score for wrong ones
        Tries to maximize reward over time

        Examples
        Training a robot to walk
        Game AI (Chess, Go, Mario bot)
        Self-driving car deciding speed and direction

**Pipeline in Machine Learning**

    Not mandatory, but very helpful
    Makes workflow organized and reliable
    Handles all preprocessing steps in a fixed order
    Reduces human mistakes and data leakage
    Keeps train and test processing consistent
    Helps build cleaner and reusable code
    Improves reliability of model results

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


**Maximum Iteration (max_iter)**

    During training, the model sees the data again and again.
    max_iter = how many times the model is allowed to learn from the data.
    More iterations → model learns longer and may perform better.
    Important when dataset is large or learning is slow.
    If max_iter is too low → model may stop early (underfit / not fully trained)
    If too high → takes more time and may overfit.

In [ ]:
pipe_lr = Pipeline([
    ("preprocessing", pre),
    ("classification", LogisticRegression(max_iter=3000, class_weight='balanced'))
])
pipe_lr.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('cata',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod']),
                                                 ('numb', 'passthrough',
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges'])])),
                ('classification',
                 LogisticRegression(class_weight='balanced', max_iter=3000))])

self study: Roc auc score

## PDSML Day-9 👉 Model Evaluation

In [ ]:
print(X_test.shape)
print(X_train.shape)

(1407, 19)
(5625, 19)


In [ ]:
print(X_test.head(10))

      gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
974   Female              0     Yes        Yes      59          Yes   
619   Female              0      No         No       7          Yes   
4289  Female              0      No         No      54          Yes   
3721  Female              0      No         No       2          Yes   
4533  Female              0     Yes         No      71          Yes   
445   Female              0      No         No      60          Yes   
5898  Female              0     Yes        Yes      33          Yes   
3387  Female              0      No         No       7          Yes   
1346  Female              0     Yes        Yes      14          Yes   
5690    Male              0      No         No      72           No   

         MultipleLines InternetService       OnlineSecurity  \
974                 No             DSL                   No   
619                Yes     Fiber optic                   No   
4289                No       

In [ ]:
print(y_test.head(10))

974     0
619     0
4289    0
3721    1
4533    0
445     1
5898    0
3387    0
1346    1
5690    0
Name: Churn, dtype: int64


In [ ]:
y_pred = pipe_lr.predict(X_test)

In [ ]:
print(y_pred[:10])

[0 1 0 0 0 1 0 0 1 0]


difference bettwen numpy arrrya and dataset/pd dataset

In [ ]:
y_proba_pred = pipe_lr.predict_proba(X_test)[:,1]

In [ ]:
print(y_proba_pred[:10])

[0.04959481 0.7866839  0.01349113 0.40202474 0.23230679 0.72305844
 0.07200374 0.34843751 0.84761562 0.04525358]


In [ ]:
from sklearn.metrics import confusion_matrix , accuracy_score

In [ ]:
cm = confusion_matrix(y_test, y_pred)

In [ ]:
cm

array([[723, 310],
       [ 76, 298]])

|          | Predicted 0 | Predicted 1 |
| -------- | ----------- | ----------- |
| Actual 0 | 723         | 310         |
| Actual 1 | 76          | 298         |

**Meaning of each value:**

    True Negative (TN) = 723 → correctly predicted 0
    False Positive (FP) = 310 → wrongly predicted 1 while actually 0
    False Negative (FN) = 76 → wrongly predicted 0 while actually 1
    True Positive (TP) = 298 → correctly predicted 1

In [ ]:
accuracy_score(y_test, y_pred)

0.7256574271499645